# Causal Inference Crash Course:
## Surrogate Models
Julian Hsu

## Overview
* **Surrogate models**: estimate the effect on a **long-term outcome** before that outcome exists, by bridging through intermediate ("surrogate") outcomes.
* Earlier decks asked *is the effect identified?* Here the treatment is randomized — identification is free — but the outcome won't arrive until **after the decision deadline**.
* We cover:
    * The waiting problem and the big idea
    * The surrogate index: two samples, three steps
    * The three assumptions, and what breaks
    * Diagnostics you can actually run
    * A simulated worked example
* Assumes the earlier decks: potential outcomes, randomized experiments, OLS, inference.

## The waiting problem
![Image](Figures/Surrogate_Figure_0.png)
* We pilot a **loyalty program** ($W_i$, randomized 50/50) for 8 weeks; the metric that matters is **3-year customer spend** ($Y_i$).
* An 8-week pilot must justify a 3-year claim by month 3 — waiting 36 months for $Y$ is an experiment design, not a decision strategy.

## Two tempting shortcuts
![Image](Figures/Surrogate_Figure_1.png)
* **Declare on the short-run metric**: assume the week-8 winner is the 3-year winner.
* **Extrapolate**: assume the week-8 effect persists.
* Both trajectories match the experiment *exactly* through week 8 — the experiment alone cannot tell them apart.

## The Big Idea
* The treatment can only change the long run **through things it changes now**.
* Call those intermediate outcomes **surrogates** $S_i$: week-8 spend, purchase frequency, engagement, &hellip;
* Two facts we *can* measure by month 3:
    * The **experiment** tells us how treatment moves the surrogates ($W \to S$).
    * **History** tells us how surrogates map into long-term outcomes ($S \to Y$).
* **Chain them**: $W \to S \to Y$.
* One clinical surrogate: Prentice (1989). Many surrogates combined into an **index**: Athey, Chetty, Imbens & Kang (2019).

# The surrogate index
Two samples, three steps, one prediction model.

## Two samples
| Sample | $W$ (treatment) | $X$ (baseline) | $S$ (week-8) | $Y$ (3-year) |
|:---|:---:|:---:|:---:|:---:|
| **Experimental** — the pilot | &#10003; | &#10003; | &#10003; | *not yet* |
| **Observational** — historical cohort | &mdash; | &#10003; | &#10003; | &#10003; |

* The historical cohort predates the program: those customers' full arcs — week-8 behavior *and* 3-year spend — are already on file.
* The pilot has the causal contrast; history has the horizon. Neither alone answers the question.

## Three steps
1. **Learn the bridge** (observational sample): fit $\hat{y}(s, x) = \hat{E}[Y \mid S=s, X=x]$.
    * A pure prediction task — OLS here, but any model that predicts well works.
2. **Score the pilot** (experimental sample): compute the **surrogate index** $\hat{Y}^{SI}_i = \hat{y}(S_i, X_i)$ — each customer's week-8 behavior converted into predicted 3-year spend.
3. **Difference it**:
$$\hat\tau_{SI} = \frac{1}{n_1}\sum_{i:\,W_i=1} \hat{Y}^{SI}_i \;-\; \frac{1}{n_0}\sum_{i:\,W_i=0} \hat{Y}^{SI}_i$$

## What it looks like
![Image](Figures/Surrogate_Figure_2.png)
* Left: history traces the $S \to Y$ map.
* Right: treatment shifts the surrogate distribution; pushed through the map, the index shifts with it. The gap between arm means is $\hat\tau_{SI}$.

## Why it works
* Randomization gives $\tau = E[Y_i \mid W_i=1] - E[Y_i \mid W_i=0]$ — but $Y$ is missing in the pilot.
* Iterated expectations: $E[Y \mid W=w] = E\big[\, E[Y \mid S, X, W=w] \;\big|\; W=w \big]$.
* **Surrogacy** deletes the inner $W$: $E[Y \mid S, X, W] = E[Y \mid S, X]$.
* **Comparability** lets history estimate that object: it is the same $E[Y \mid S, X]$ the observational sample identifies — our $\hat{y}(s,x)$.
* So averaging $\hat{y}(S_i, X_i)$ by arm recovers each arm's mean of $Y$ — without ever observing $Y$ in the pilot.

# Assumptions
Three. The first is the one that should keep you up at night.

## Assumption 1: surrogacy
$$W_i \;\perp\; Y_i \;\big|\; S_i, X_i$$
* Given the surrogates (and baseline), treatment status carries **no extra information** about the long-run outcome: every causal path from $W$ to $Y$ passes through $S$.
* For a *single* surrogate this is the **Prentice criterion** — and it is almost always false.
* The modern move: make $S$ a **rich bundle**. The more of the treatment's short-run footprint you record, the less room for a bypassing channel.
* Untestable at the horizon you care about — and a merely *predictive* surrogate is not enough: see [**Appendix — the surrogate paradox**](#/appendix-the-surrogate-paradox).

## Assumption 2: comparability
* The $S \to Y$ map must be the **same** in both samples: $Y_i \perp G_i \mid S_i, X_i$, where $G_i$ flags which sample unit $i$ belongs to.
* Threats:
    * **Different populations** — history covers all customers, the pilot only app users.
    * **Different regimes** — history spans a boom, the pilot runs into a recession.
    * **The program rewires the map** — pull-forward: promo-induced week-8 spend that cannibalizes year-2 spend predicts $Y$ differently than organic week-8 spend does.
* Plus **overlap**: the pilot's $(S, X)$ values must actually occur in history — the bridge cannot be extrapolated.

## Assumption 3: a clean experiment
* $W_i \perp \big(Y_i(1), Y_i(0), S_i(1), S_i(0)\big)$ — free when the pilot is randomized; in an observational "pilot" you are stacking unconfoundedness (deck 2) *on top of* surrogacy.

| Assumption | What it buys | Main threat |
|:---|:---|:---|
| 1. Surrogacy | $W$ drops out given $(S, X)$ | a channel no surrogate records |
| 2. Comparability | history estimates the map | population / regime shift, pull-forward |
| 3. Clean experiment | the causal $W \to S$ contrast | rarely an issue in a true A/B test |

## One surrogate is fragile
![Image](Figures/Surrogate_Figure_3.png)
* Each surrogate alone carries only part of the effect; the estimate climbs toward the truth as the bundle fills out.
* This is why Athey et al. call it a surrogate **index** — their application bridges through *several quarters* of employment history, not one number.

## When surrogacy fails
![Image](Figures/Surrogate_Figure_4.png)
* Give the treatment a channel no surrogate records (size $\theta$): the index misses **exactly** that channel — bias $=\theta$, silently, with tight confidence intervals around the wrong number.
* The direction is reasoned about, not tested: a missed channel working *with* the treatment means the index **understates**; against, **overstates**. Athey et al. (2019) derive bounds under partial surrogacy.

## Diagnostics
* **Backtest on finished experiments**: for past A/B tests whose long-run outcomes have since matured, compute the week-8 index and compare it to the realized effect. *The* industry validation.
* **Horizon laddering**: validate the 8-week index against 6-month outcomes (already observable today), the 6-month index against 1-year, &hellip; — surrogacy failures usually surface early.
* **Leave-one-surrogate-out**: if dropping any single surrogate moves $\hat\tau_{SI}$ materially, the bundle is too thin to trust.
* **Re-fit the bridge on the pilot's control arm** as its early outcomes trickle in and compare with history's bridge — a direct comparability check.

# Worked example
Simulated pilot, so we know the answer ($\tau = 5$).

## The setup
* Pilot: $n = 2{,}000$ customers randomized 50/50, run for 8 weeks. Historical cohort: $n = 10{,}000$ customers with 3-year spend on file.
* Three week-8 surrogates: **early spend**, **purchase frequency**, **engagement**; one baseline covariate (prior spend). True 3-year effect $\tau = 5$.
* Two scenarios:
    1. **Valid** — the treatment reaches 3-year spend *only* through the three surrogates.
    2. **Hidden channel** — an extra effect of $\theta = 1.5$ bypasses them (true effect $= 6.5$).
* Companion notebook with all code [here](https://github.com/shoepaladin/causalinference_crashcourse/blob/main/Notebooks/8%20Surrogate%20Models.ipynb).

## Four estimators
1. **Short-run difference**: the arm gap in week-8 spend, reported as if it were the 3-year effect.
2. **Single surrogate**: the surrogate index built from week-8 spend alone (Prentice-style).
3. **Surrogate index**: the index from all three surrogates plus the baseline covariate (OLS bridge).
4. **Oracle — wait 3 years**: the pilot's actual long-run difference. The benchmark the deadline forbids.

## Bias
* Mean bias $(\hat\tau - \tau)$ over 500 simulations:

| Estimator | Valid ($\tau = 5$) | Hidden channel ($\tau = 6.5$) |
|:---|:---:|:---:|
| Short-run difference | -3.00 | -4.50 |
| Single surrogate | -0.98 | -2.48 |
| Surrogate index | -0.01 | -1.51 |
| Oracle (wait 3 years) | +0.00 | +0.00 |

* With a rich bundle and valid surrogacy the index is clean; under the hidden channel it is off by **exactly** $\theta = 1.5$.
* One-draw picture with confidence intervals: [**Appendix — the estimates, drawn**](#/appendix-the-estimates-drawn).

## Inference: the index is a prediction, not data
* Two stages carry noise: the experiment *and* the estimated bridge $\hat{y}(\cdot)$.
* A **plug-in** CI (t-test treating the index as data) ignores stage 1; the **bootstrap** resamples both samples and refits the bridge every draw.

| 95% CI for the surrogate index | Coverage, $n_{obs} = 10{,}000$ | Coverage, $n_{obs} = 500$ |
|:---|:---:|:---:|
| Plug-in | 0.95 | 0.85 |
| Bootstrap (both stages) | 0.94 | 0.91 |

* A large historical sample makes the shortcut nearly harmless; a thin one flatters your precision. Athey et al. (2019) provide analytic (influence-function) standard errors.

## A precision bonus
![Image](Figures/Surrogate_Figure_6.png)
* In the worked example the index's SD (0.25) *beats* the oracle's (0.31): the index averages away the part of $Y$ that surrogates cannot predict, while waiting keeps all of that noise.
* The lower the surrogates' $R^2$ for $Y$, the bigger the precision advantage — and the more nervous surrogacy should make you. Precision is not validity.

## Conclusion
* The surrogate index chains the two things you can measure by the deadline: the experiment's $W \to S$ and history's $S \to Y$.
* Pros: an answer **when the decision is due**; stage 1 is ordinary supervised prediction; often *tighter* CIs than waiting for the outcome.
* Cons: surrogacy is untestable at the horizon that matters; a bypassed channel biases silently; comparability and overlap constrain which history you may borrow.
* Reach for it when the decision can't wait — then **backtest against finished experiments** and keep logging richer surrogates.

# Appendix Slides

## Appendix: the estimates, drawn
![Image](Figures/Surrogate_Figure_5.png)
* One draw of the worked example (valid scenario), 95% bootstrap CIs.
* The short-run difference and the single surrogate are *precisely wrong* — tight intervals far from the truth. The index and the oracle both cover it; the index is a bit tighter.

## Appendix: the surrogate score
* The index is one of **two mirror-image estimators** in Athey et al. (2019).
* **Surrogate index**: work in the *experimental* sample, replacing missing $Y$ with predicted $\hat{y}(S, X)$.
* **Surrogate score**: work in the *observational* sample, where $Y$ is real, weighting each unit by how "treated" its surrogate bundle looks:
$$\hat{q}(s, x) = \hat{P}(W=1 \mid S=s, X=x) \text{ estimated in the experimental sample,}$$
then average observational $Y_i$ with weights $\propto \hat{q}(S_i,X_i)$ for the treated mean and $\propto 1-\hat{q}(S_i,X_i)$ for the control mean — a propensity-style reweighting where surrogates play the role of covariates.
* Combining both nuisance models yields a **doubly-robust** estimator: consistent if *either* the bridge or the score is well specified, with better efficiency (Chen & Ritzwoller 2023; Kallus & Mao 2020).

## Appendix: bias under partial surrogacy
* Decompose the true effect as $\tau = \tau_S + \theta$: the part transmitted through recorded surrogates, plus the bypass.
* Under comparability and a clean experiment, $E[\hat\tau_{SI}] = \tau_S$, so
$$\text{bias}(\hat\tau_{SI}) = -\,\theta .$$
* Reasoning about $\theta$:
    * Sign: does the unrecorded channel plausibly work with or against the recorded ones?
    * Size: Athey et al. (2019) bound the bias using the treatment's residual association with $Y$ given $S$ — the smaller the room left after conditioning on a rich $S$, the tighter the bound.
* Practical translation: every additional orthogonal surrogate you log shrinks the worst case.

## Appendix: the surrogate paradox
* It is possible for the treatment to **raise every surrogate**, every surrogate to **positively predict** $Y$ — and the treatment to **lower** $Y$ (VanderWeele 2013).
* How: an unobserved variable raises $S$ but lowers $Y$, or a bypassing negative direct effect — predictive correlation survives, the causal chain does not.
* The cautionary classic: in the CAST trial, anti-arrhythmic drugs suppressed arrhythmias (the accepted surrogate for cardiac death) and *increased* mortality.
* Moral: a surrogate that merely *predicts* is not a surrogate that *transmits*. Surrogacy is the conditional-independence statement $W \perp Y \mid S, X$ — check paths, not correlations.

## Appendix: surrogates vs. mediation vs. principal stratification
* **Mediation analysis** decomposes an observed total effect into channels; it needs sequential ignorability *and* you already have $Y$. Different goal: explanation, not forecasting.
* **Principal stratification** (Frangakis & Rubin 2002) defines "principal surrogacy" via joint potential values of $S$ — conceptually cleaner, much harder to estimate.
* The **surrogate index** claims less than either: it never interprets the $S \to Y$ coefficients causally — the bridge is *only a prediction model*. What it must swallow instead is total mediation: **no bypass at all**.
* Corollary: don't read $\hat{y}$'s coefficients as "the value of engagement"; they are predictive weights, nothing more.

## Appendix: practical guidance
* Log **many, cheap, hard-to-game** surrogates: breadth of behavior (frequency, recency, category breadth, engagement, support contacts) beats one clever metric.
* **Pre-register the bridge model** — sample, features, algorithm — before unblinding the pilot, or the index becomes a metric-shopping exercise.
* Beware **Goodhart's law**: once teams target the index, the historical $S \to Y$ map is exactly what breaks (a comparability violation you caused).
* Maintain a **backtest library**: every past experiment whose long-run outcomes have matured is a free validation point for the current index.
* Refresh the bridge on rolling cohorts; regime changes (pandemic, price overhaul, new product mix) invalidate old maps.

## Literature Review of Related Papers
* **Foundational (biostatistics)**
    * Prentice (1989). "Surrogate endpoints in clinical trials: Definition and operational criteria." *Statistics in Medicine.* [link](https://onlinelibrary.wiley.com/doi/10.1002/sim.4780080407)
    * Freedman, Graubard, Schatzkin (1992). "Statistical validation of intermediate endpoints for chronic diseases." *Statistics in Medicine.* [link](https://onlinelibrary.wiley.com/doi/10.1002/sim.4780110207)
    * Frangakis & Rubin (2002). "Principal Stratification in Causal Inference." *Biometrics.* [link](https://onlinelibrary.wiley.com/doi/10.1111/j.0006-341X.2002.00021.x)
    * VanderWeele (2013). "Surrogate measures and consistent surrogates." *Biometrics.* [link](https://onlinelibrary.wiley.com/doi/10.1111/biom.12071)
* **The surrogate index & econometrics**
    * Athey, Chetty, Imbens, Kang (2019). "The Surrogate Index: Combining Short-Term Proxies to Estimate Long-Term Treatment Effects More Rapidly and Precisely." NBER WP 26463. [link](https://www.nber.org/papers/w26463)
    * Athey, Chetty, Imbens (2020). "Combining Experimental and Observational Data to Estimate Treatment Effects on Long Term Outcomes." [link](https://arxiv.org/abs/2006.09676)
    * Kallus & Mao (2020). "On the role of surrogates in the efficient estimation of treatment effects with limited outcome data." [link](https://arxiv.org/abs/2003.12408)
    * Chen & Ritzwoller (2023). "Semiparametric Estimation of Long-Term Treatment Effects." *Journal of Econometrics.* [link](https://arxiv.org/abs/2107.14405)
* **Applications & industry practice**
    * Hohnhold, O'Brien, Tang (2015). "Focusing on the Long-term: It's Good for Users and Business." *KDD.* [link](https://research.google/pubs/pub43887/)
    * Yang, Eckles, Dhillon, Aral (2024). "Targeting for Long-Term Outcomes." *Management Science.* [link](https://arxiv.org/abs/2010.15835)

In [ ]:
# --- Figure generation (run this notebook to regenerate Figures/Surrogate_Figure_*.png) ---
# All figures come from a simulated loyalty-program pilot: an 8-week experiment
# whose outcome of record (3-year spend) is bridged through week-8 surrogates.
import os
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(20260707)
TAU = 5.0                                # true 3-year effect through the surrogates
ACCENT, ACCENT2, CTRLC = "#0e7490", "#0f766e", "#94a3b8"
FIGDIR = os.path.join(os.getcwd(), "Figures")
os.makedirs(FIGDIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 11, "axes.grid": True, "grid.alpha": .25})

# Three week-8 surrogates: early spend, purchase frequency, engagement.
# W moves surrogate k by C[k]; surrogate k is worth G[k] of 3-year spend, so tau = C @ G = 5.
A = np.array([10.0, 4.0, 2.0])           # surrogate intercepts
B = np.array([1.0, 0.8, 0.6])            # loadings on latent loyalty
C = np.array([2.0, 1.0, 1.0])            # effect of W on each surrogate
G = np.array([1.5, 1.0, 1.0])            # value of each surrogate in 3-year spend
S_SD = np.array([1.5, 1.0, 1.0])


def gen(n, experimental=True, theta=0.0, sd_y=4.0, seed=None):
    """One sample. Experimental: W randomized 50/50 and Y is 3 years out (unseen at
    decision time). Observational: historical cohort, program didn't exist (W=0), Y on file.
    theta is a direct effect of W on Y that bypasses every surrogate (surrogacy violation)."""
    rng = np.random.default_rng(seed) if seed is not None else RNG
    x = rng.normal(0, 1, n)                                  # baseline: prior spend (standardized)
    loyal = 0.5 * x + rng.normal(0, 1.15 if not experimental else 1.0, n)
    w = rng.integers(0, 2, n).astype(float) if experimental else np.zeros(n)
    s = A + B * loyal[:, None] + C * w[:, None] + rng.normal(0, S_SD, (n, 3))
    y = 20.0 + s @ G + 2.0 * x + theta * w + rng.normal(0, sd_y, n)
    return x, w, s, y


def fit_index(x_o, s_o, y_o, cols=(0, 1, 2)):
    """Step 1: the bridge — OLS of Y on (S, X) in the observational sample."""
    Z = np.column_stack([np.ones_like(y_o), s_o[:, list(cols)], x_o])
    beta, *_ = np.linalg.lstsq(Z, y_o, rcond=None)
    return beta, cols


def score_index(fit, x_e, s_e):
    """Step 2: the surrogate index for each experimental unit."""
    beta, cols = fit
    Z = np.column_stack([np.ones(len(x_e)), s_e[:, list(cols)], x_e])
    return Z @ beta


def tau_si(fit, x_e, w_e, s_e):
    """Step 3: difference the index between arms."""
    idx = score_index(fit, x_e, s_e)
    return idx[w_e == 1].mean() - idx[w_e == 0].mean()


def _save(fig, name):
    fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, name), bbox_inches="tight"); plt.close(fig)

In [ ]:
# Figure 0 — the waiting problem: what is observable by the decision deadline
fig, ax = plt.subplots(figsize=(8.4, 3.0))
bars = [("Historical cohort\n(3-year spend observed)", -40, 40, CTRLC),
        ("Pilot experiment (8 weeks)", 0, 2, ACCENT),
        ("Pilot's 3-year spend", 0, 38, "#e2e8f0")]
for i, (label, start, length, color) in enumerate(bars):
    yy = 2 - i
    ax.broken_barh([(start, length)], (yy - 0.28, 0.56), facecolors=color,
                   edgecolor="#334155", linewidth=0.8)
    ax.text(start - 1.2, yy, label, ha="right", va="center", fontsize=9.5)
ax.annotate("observable only here,\n36 months too late", xy=(38, 0), xytext=(20, -0.85),
            fontsize=9, color="#7c2d12", arrowprops=dict(arrowstyle="->", color="#7c2d12"))
ax.axvline(3, color="#7c2d12", ls="--", lw=1.4)
ax.text(3.4, 2.55, "decision deadline (month 3)", color="#7c2d12", fontsize=9.5)
ax.set_xlim(-42, 44); ax.set_ylim(-1.3, 3.0)
ax.set_xlabel("Months relative to pilot launch")
ax.set_yticks([]); ax.grid(axis="y", alpha=0)
_save(fig, "Surrogate_Figure_0.png")

In [ ]:
# Figure 1 — the same week-8 result, two very different 3-year stories
m = np.linspace(0, 36, 200)
habit = (1 - np.exp(-m / 9.5)) / (1 - np.exp(-2 / 9.5))   # effect builds; equals 1.0 at month 2
novelty = np.exp(-(m - 2) / 6.0)                          # effect fades after the novelty wears off
novelty[m < 2] = m[m < 2] / 2                             # ramps up to the same week-8 point
fig, ax = plt.subplots(figsize=(7.6, 3.4))
ax.plot(m, habit, color=ACCENT2, lw=2.2, label="habit builds")
ax.plot(m, novelty, color="#b45309", lw=2.2, label="novelty fades")
ax.axhline(1.0, color="#334155", ls=":", lw=1.6, label='naive: "the week-8 effect persists"')
ax.axvline(2, color="#7c2d12", ls="--", lw=1.2)
ax.text(2.3, 4.6, "week 8:\nall the experiment sees", fontsize=9, color="#7c2d12")
ax.set_xlabel("Months since launch"); ax.set_ylabel("Effect on monthly spend ($)")
ax.legend(loc="center right", fontsize=9)
_save(fig, "Surrogate_Figure_1.png")

In [ ]:
# Figure 2 — the estimator, drawn: history teaches S -> Y; the experiment shifts S
x_o, w_o, s_o, y_o = gen(10000, experimental=False, seed=11)
x_e, w_e, s_e, y_e = gen(2000, experimental=True, seed=12)
fit = fit_index(x_o, s_o, y_o)
idx = score_index(fit, x_e, s_e)
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.4))
ax = axes[0]
sub = RNG.choice(len(y_o), 900, replace=False)
ax.scatter(s_o[sub, 0], y_o[sub], s=8, color=CTRLC, alpha=0.45)
xs = np.linspace(s_o[:, 0].min(), s_o[:, 0].max(), 50)
b = np.polyfit(s_o[:, 0], y_o, 1)
ax.plot(xs, np.polyval(b, xs), color=ACCENT, lw=2.4)
ax.set_xlabel("Week-8 spend  $S_1$"); ax.set_ylabel("3-year spend  $Y$")
ax.set_title("Historical cohort: learn  $\\hat{y}(S,X) = E[Y\\,|\\,S,X]$", fontsize=10)
ax = axes[1]
for arm, col, lab in ((0, "#0f172a", "control"), (1, ACCENT2, "treatment")):
    vals = idx[w_e == arm]
    ax.hist(vals, bins=35, density=True, alpha=0.55, color=col, label=lab)
    ax.axvline(vals.mean(), color=col, lw=2.2)
gap = idx[w_e == 1].mean() - idx[w_e == 0].mean()
ax.annotate(f"$\\hat\\tau_{{SI}}$ = {gap:.1f}",
            xy=(idx[w_e == 1].mean(), ax.get_ylim()[1] * 0.92),
            xytext=(idx[w_e == 1].mean() + 3, ax.get_ylim()[1] * 0.9),
            fontsize=11, color="#7c2d12",
            arrowprops=dict(arrowstyle="->", color="#7c2d12"))
ax.set_xlabel("Surrogate index  $\\hat{y}(S_i, X_i)$"); ax.set_ylabel("Density")
ax.set_title("Experiment: the index shifts under treatment", fontsize=10)
ax.legend(fontsize=9)
_save(fig, "Surrogate_Figure_2.png")

In [ ]:
# Figure 3 — one surrogate is fragile: mean estimate vs. the set of surrogates in the index
sets = [(0,), (0, 1), (0, 1, 2)]
labels = ["$S_1$ only\n(week-8 spend)", "$S_1, S_2$", "$S_1, S_2, S_3$\n(full set)"]
ests = {k: [] for k in range(len(sets))}
for r in range(400):
    x_o, w_o, s_o, y_o = gen(10000, experimental=False, seed=1000 + r)
    x_e, w_e, s_e, y_e = gen(2000, experimental=True, seed=5000 + r)
    for k, cols in enumerate(sets):
        ests[k].append(tau_si(fit_index(x_o, s_o, y_o, cols), x_e, w_e, s_e))
means = [np.mean(ests[k]) for k in range(len(sets))]
fig, ax = plt.subplots(figsize=(7.2, 3.4))
bars = ax.bar(labels, means, color=[CTRLC, "#5eead4", ACCENT], edgecolor="#334155", width=0.55)
ax.axhline(TAU, color="#7c2d12", ls="--", lw=1.6)
ax.text(2.45, TAU + 0.08, f"truth $\\tau$ = {TAU:.0f}", color="#7c2d12", fontsize=10, ha="right")
for bar, v in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.08, f"{v:.2f}", ha="center", fontsize=10)
ax.set_ylabel("Mean surrogate-index estimate")
ax.set_ylim(0, TAU + 1.0)
_save(fig, "Surrogate_Figure_3.png")

In [ ]:
# Figure 4 — a hidden channel biases the index by exactly the bypassed effect
thetas = np.linspace(0, 2.5, 6)
si_means, truths = [], []
for th in thetas:
    vals = []
    for r in range(200):
        x_o, w_o, s_o, y_o = gen(10000, experimental=False, seed=2000 + r)
        x_e, w_e, s_e, y_e = gen(2000, experimental=True, theta=th, seed=8000 + r)
        vals.append(tau_si(fit_index(x_o, s_o, y_o), x_e, w_e, s_e))
    si_means.append(np.mean(vals)); truths.append(TAU + th)
fig, ax = plt.subplots(figsize=(7.4, 3.4))
ax.plot(thetas, truths, color="#7c2d12", lw=2.2, marker="o", ms=4, label="true effect  $\\tau + \\theta$")
ax.plot(thetas, si_means, color=ACCENT, lw=2.2, marker="o", ms=4, label="surrogate index estimate")
ax.fill_between(thetas, si_means, truths, color="#fecaca", alpha=0.5)
ax.text(1.7, TAU + 0.9, "bias = the hidden channel", fontsize=10, color="#b91c1c", rotation=11)
ax.set_xlabel("Direct effect $\\theta$ that bypasses the surrogates")
ax.set_ylabel("3-year effect ($)")
ax.legend(fontsize=9, loc="upper left")
_save(fig, "Surrogate_Figure_4.png")

In [ ]:
# Figure 5 — worked example, one draw: estimates and 95% bootstrap CIs
x_o, w_o, s_o, y_o = gen(10000, experimental=False, seed=42)
x_e, w_e, s_e, y_e = gen(2000, experimental=True, seed=43)

def est_all(x_o, s_o, y_o, x_e, w_e, s_e, y_e):
    naive = s_e[w_e == 1, 0].mean() - s_e[w_e == 0, 0].mean()
    single = tau_si(fit_index(x_o, s_o, y_o, (0,)), x_e, w_e, s_e)
    full = tau_si(fit_index(x_o, s_o, y_o), x_e, w_e, s_e)
    oracle = y_e[w_e == 1].mean() - y_e[w_e == 0].mean()
    return naive, single, full, oracle

point = est_all(x_o, s_o, y_o, x_e, w_e, s_e, y_e)
names = ["Short-run difference\n(week-8 spend)", "Single surrogate",
         "Surrogate index\n(all three)", "Oracle: wait 3 years"]
rng = np.random.default_rng(7)
cis = []
for j in range(4):
    vals = []
    for _ in range(300):
        io = rng.integers(0, len(y_o), len(y_o)); ie = rng.integers(0, len(y_e), len(y_e))
        vals.append(est_all(x_o[io], s_o[io], y_o[io], x_e[ie], w_e[ie], s_e[ie], y_e[ie])[j])
    cis.append(np.percentile(vals, [2.5, 97.5]))
fig, ax = plt.subplots(figsize=(7.8, 3.4))
ypos = np.arange(4)[::-1]
for i, (nm, pt, ci) in enumerate(zip(names, point, cis)):
    col = ACCENT if "index" in nm else ("#0f172a" if "Oracle" in nm else CTRLC)
    ax.errorbar(pt, ypos[i], xerr=[[pt - ci[0]], [ci[1] - pt]], fmt="o", color=col,
                capsize=4, ms=7, lw=2)
ax.axvline(TAU, color="#7c2d12", ls="--", lw=1.6)
ax.text(TAU + 0.06, 3.28, "truth", color="#7c2d12", fontsize=10)
ax.set_yticks(ypos); ax.set_yticklabels(names, fontsize=9.5)
ax.set_xlabel("Estimated 3-year effect ($), one draw, 95% bootstrap CI")
_save(fig, "Surrogate_Figure_5.png")

In [ ]:
# Figure 6 — precision: the index is often *tighter* than waiting for Y
sd_ys = [1.0, 2.0, 4.0, 7.0, 11.0]      # noise in Y traces out the surrogates' R^2
r2s, ratios = [], []
for sdy in sd_ys:
    t_si, t_or = [], []
    for r in range(300):
        x_o, w_o, s_o, y_o = gen(10000, experimental=False, sd_y=sdy, seed=3000 + r)
        x_e, w_e, s_e, y_e = gen(2000, experimental=True, sd_y=sdy, seed=9000 + r)
        t_si.append(tau_si(fit_index(x_o, s_o, y_o), x_e, w_e, s_e))
        t_or.append(y_e[w_e == 1].mean() - y_e[w_e == 0].mean())
    x_o, w_o, s_o, y_o = gen(20000, experimental=False, sd_y=sdy, seed=99)
    fitb = fit_index(x_o, s_o, y_o)
    resid = y_o - score_index(fitb, x_o, s_o)
    r2s.append(1 - resid.var() / y_o.var())
    ratios.append(np.std(t_si) / np.std(t_or))
fig, ax = plt.subplots(figsize=(7.4, 3.4))
ax.plot(r2s, ratios, color=ACCENT, lw=2.2, marker="o")
ax.axhline(1.0, color="#334155", ls=":", lw=1.5)
ax.text(r2s[1], 1.03, "same width as waiting for $Y$", fontsize=9, color="#334155")
ax.set_xlabel("$R^2$ of the surrogates for the 3-year outcome (historical data)")
ax.set_ylabel("SE(surrogate index) / SE(wait for $Y$)")
ax.set_ylim(0, 1.25)
_save(fig, "Surrogate_Figure_6.png")